In [ ]:
# Install necessary libraries
!pip install bertopic sentence-transformers umap-learn

In [ ]:
!pip uninstall -y scikit-learn umap-learn hdbscan
!pip install scikit-learn==1.3.2 umap-learn==0.5.5 hdbscan==0.8.33 bertopic==0.16.4


In [1]:
import polars as pl
import s3fs
import pyarrow.dataset as ds  

# --- CONFIGURATION ---
S3_REVIEW_PATH = "s3://curated-review-data/reviews/appliances_reviews.parquet"  
S3_SAMPLE_OUTPUT = "s3://curated-review-data/risk_training_sample_appliance.parquet"

def create_risk_training_sample_robust(source_path, sample_size=200000):
    print(f"Connecting to S3 via PyArrow Dataset: {source_path}...")
    
    fs = s3fs.S3FileSystem()
    dataset = ds.dataset(source_path, filesystem=fs, format="parquet")
    
    # 2. Convert to Polars LazyFrame
    # This inherits PyArrow's schema flexibility while keeping Polars' speed
    q = pl.scan_pyarrow_dataset(dataset)
    

    risk_query = (
        q.with_columns(pl.col("rating").cast(pl.Float64)) # Unifies type
         .filter(pl.col("rating") <= 3.0)
         .select(["final_text", "rating", "asin", "parent_asin", "timestamp"])
    )
    
    print("Collecting negative reviews into memory (Robust Mode)...")
    df_negative = risk_query.collect() 
    
    print(f"Total Negative Reviews Found: {df_negative.height}")
    
    # 4. Stratified Sampling
    print(f"Sampling {sample_size} rows...")
    df_sample = df_negative.sample(n=sample_size, shuffle=True, seed=42)
    
    return df_sample

df_train_sample = create_risk_training_sample_robust(S3_REVIEW_PATH)

# Save sample
df_train_sample.write_parquet(S3_SAMPLE_OUTPUT)

print("✅ Stage 1 Complete. Sample Data Ready.")
print(df_train_sample.head())

Connecting to S3 via PyArrow Dataset: s3://curated-review-data/reviews/appliances_reviews.parquet...
Total Negative Reviews Found: 404645
Sampling 200000 rows...
✅ Stage 1 Complete. Sample Data Ready.
shape: (5, 5)
┌─────────────────────────────────┬────────┬────────────┬─────────────┬───────────────┐
│ final_text                      ┆ rating ┆ asin       ┆ parent_asin ┆ timestamp     │
│ ---                             ┆ ---    ┆ ---        ┆ ---         ┆ ---           │
│ str                             ┆ f64    ┆ str        ┆ str         ┆ i64           │
╞═════════════════════════════════╪════════╪════════════╪═════════════╪═══════════════╡
│ ehhh didnt purchase amazon sti… ┆ 2.0    ┆ B097K9FZB2 ┆ B097KBJF48  ┆ 1663560197372 │
│ horrible quality broke one use… ┆ 1.0    ┆ B086GTPXPN ┆ B09YV4R5ZC  ┆ 1616696580197 │
│ waste money bought february  m… ┆ 1.0    ┆ B08N4GQYXQ ┆ B0C5DBKGTB  ┆ 1681570157390 │
│ no ice never used another filt… ┆ 2.0    ┆ B01GAAYTGY ┆ B0BCW97VD9  ┆ 159028249

In [2]:
import hdbscan.hdbscan_
import sklearn.utils.validation
import polars as pl
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer

df_fit = pl.read_parquet(S3_SAMPLE_OUTPUT)
docs_fit = df_fit["final_text"].to_list()

print("Generating Embeddings...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
emb_fit = embedder.encode(
    docs_fit, 
    batch_size=32, 
    show_progress_bar=True, 
    convert_to_numpy=True
).astype(np.float16)


umap_model = UMAP(
    n_neighbors=15, 
    n_components=5, 
    min_dist=0.0, 
    metric="cosine", 
    random_state=42
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=80,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True 
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=15,
    max_features=50_000
)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)

# --- 4. TRAIN ---
print("Fitting BERTopic with HDBSCAN...")
topic_model.fit(docs_fit, embeddings=emb_fit)
print("✅ Model Trained Successfully with HDBSCAN!")

# --- 5. SAVE ---
topic_model.save("risk_topic_model", serialization="safetensors", save_embedding_model=True)
print("--- Top 10 Risk Topics ---")
print(topic_model.get_topic_info().head(20))

Generating Embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/6250 [00:00<?, ?it/s]

2026-01-30 07:13:49,841 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic with HDBSCAN...


2026-01-30 07:20:02,920 - BERTopic - Dimensionality - Completed ✓
2026-01-30 07:20:02,924 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-30 07:20:18,622 - BERTopic - Cluster - Completed ✓
2026-01-30 07:20:18,656 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-01-30 07:20:32,813 - BERTopic - Representation - Completed ✓


✅ Model Trained Successfully with HDBSCAN!
--- Top 10 Risk Topics ---
    Topic  Count                                      Name  \
0      -1  75177              -1_filter_water_work_product   
1       0   5752          0_ice_ice maker_maker_making ice   
2       1   3411  1_stopped working_working_stopped_months   
3       2   3273                2_wick_smell_mold_humidity   
4       3   2772                3_belt_rollers_pulley_drum   
5       4   2563               4_ice_cubes_melts_ice cubes   
6       5   2408    5_humidity_accurate_temperature_sensor   
7       6   2237      6_paper_coffee_paper filters_filters   
8       7   2176           7_waste_waste money_money_price   
9       8   2143               8_oven_cooktop_burner_range   
10      9   1991      9_star_stars_zero stars_star product   
11     10   1971       10_install_easy install_easy_lasted   
12     11   1855     11_coffee_grounds_weak_coffee grounds   
13     12   1806              12_pods_pod_espresso_capsule   


In [5]:

# --- CONFIGURATION ---
S3_INPUT_PATTERN = "s3://curated-review-data/reviews/appliances_reviews.parquet" # Your specific file
S3_OUTPUT_FILE = "s3://curated-review-data/processed/appliances_reviews_with_risk.parquet"

# --- STEP 1: DEFINE RISK LABELS (Appliances) ---
# Mapped from BERTopic Image Analysis (image_90bbf3.png)
risk_map = {
    -1: "General Risk",
    0: "Ice Maker Failure",              # "ice maker, making ice, stopped"
    1: "Premature Failure (General)",    # "stopped working, months, died"
    2: "Mold/Smell Issue (Humidifiers)", # "wick, smell, mold, humidity"
    3: "Mechanical Failure (Dryers)",    # "belt, rollers, pulley, drum"
    4: "Cooling Failure (Ice/Freezer)",  # "ice cubes, melts"
    5: "Sensor Accuracy (Humidity/Temp)",# "humidity, accurate, sensor"
    6: "Consumable Issue (Filters)",     # "paper, coffee, filters"
    7: "Price/Value Complaint",          # "waste money, overpriced"
    8: "Heating Element (Ovens/Stoves)", # "oven, cooktop, burner, range"
    9: "General Low Rating",             # "zero stars, star product"
    10: "Installation Difficulty",       # "install, easy install, lasted"
    11: "Brewing Issue (Coffee)",        # "coffee grounds, weak"
    12: "Pod/Capsule Compatibility",     # "pods, pod, espresso, capsule"
    13: "Venting/Duct Issue",            # "vent, duct, dryer, wall"
    14: "Language (Non-English)",        # "la, el, lo, al"
    15: "Taste/Water Quality",           # "taste, tastes, water taste"
    16: "Heating Element (Dryers)",      # "dryer, element, heating"
    17: "Noise Level (Loud)",            # "loud, ice, noise, making"
    18: "Sizing/Fit Issue"               # "size, wrong size, small"
}

print("✅ Appliances Risk Map Created.")

✅ Appliances Risk Map Created.


In [6]:
import polars as pl
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

print("Loading Models...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic.load("risk_topic_model", embedding_model=embedding_model)

print(f"Reading data from {S3_INPUT_PATTERN}...")
df = pl.read_parquet(S3_INPUT_PATTERN)

print("Splitting & Sampling 50% of Data...")

df_sampled = df.sample(fraction=0.5, seed=42)

df_neg = df_sampled.filter(pl.col("rating") <= 3.0)
df_pos = df_sampled.filter(pl.col("rating") > 3.0)

print(f"Negative Rows (To Infer): {df_neg.height}")
print(f"Positive Rows (Auto-Safe): {df_pos.height}")

if df_neg.height > 0:
    docs = df_neg["final_text"].to_list()
    print("⏳ Starting Inference on Negatives...")
    topics, _ = topic_model.transform(docs)
    print("✅ Inference Finished!")

    print("Mapping topics to labels...")
    risk_series = pl.Series("predicted_topic", topics)
    risk_column = risk_series.replace(risk_map, default="General Risk").alias("risk_category")
    
    df_neg_tagged = df_neg.with_columns([
        risk_series,
        risk_column,
        pl.lit(1).alias("is_high_risk") # Flag = 1
    ])

print("Tagging Positives...")
df_pos_tagged = df_pos.with_columns([
    pl.lit(-2).alias("predicted_topic").cast(pl.Int64), # -2 = Safe ID
    pl.lit("Safe/No Risk").alias("risk_category"),
    pl.lit(0).alias("is_high_risk") # Flag = 0
])

# --- 3. COMBINE & SAVE ---
print("Combining Data...")
if df_neg.height > 0:
    df_final = pl.concat([df_neg_tagged, df_pos_tagged])
else:
    df_final = df_pos_tagged

print(f"Saving {df_final.height} rows to {S3_OUTPUT_FILE}...")
df_final.write_parquet(S3_OUTPUT_FILE)
print("🔥 Process Complete & Saved.")
print(df_final.select(["rating", "risk_category", "is_high_risk"]).head())

Loading Models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reading data from s3://curated-review-data/reviews/appliances_reviews.parquet...
Splitting & Sampling 50% of Data...
Negative Rows (To Infer): 202096
Positive Rows (Auto-Safe): 786946
⏳ Starting Inference on Negatives...


Batches:   0%|          | 0/6316 [00:00<?, ?it/s]

2026-01-30 08:24:33,313 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


✅ Inference Finished!
Mapping topics to labels...
Tagging Positives...
Combining Data...
Saving 989042 rows to s3://curated-review-data/processed/appliances_reviews_with_risk.parquet...


/tmp/ipykernel_2485/3867816464.py:30: DeprecationWarning:

the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)



🔥 Process Complete & Saved.
shape: (5, 3)
┌────────┬───────────────┬──────────────┐
│ rating ┆ risk_category ┆ is_high_risk │
│ ---    ┆ ---           ┆ ---          │
│ f64    ┆ str           ┆ i32          │
╞════════╪═══════════════╪══════════════╡
│ 1.0    ┆ General Risk  ┆ 1            │
│ 1.0    ┆ General Risk  ┆ 1            │
│ 1.0    ┆ General Risk  ┆ 1            │
│ 1.0    ┆ General Risk  ┆ 1            │
│ 2.0    ┆ General Risk  ┆ 1            │
└────────┴───────────────┴──────────────┘


In [27]:
import polars as pl

# --- CONFIGURATION ---
# 1. The Universe (All Appliances Products)
S3_RAW_APPLIANCES_REVIEWS = "s3://curated-review-data/reviews/musical_instruments_reviews.parquet"

# 2. The AI Results (The 50% sample we analyzed)
S3_TAGGED_REVIEWS = "s3://curated-review-data/processed/musical_instruments_with_risk.parquet"

# 3. Final Output (Lean KPI Table)
S3_FINAL_OUTPUT = "s3://curated-review-data/processed/kpis only/musical_risk_kpis_only.csv"

def create_lean_kpi_table():
    print(f"1. Loading Product Universe from {S3_RAW_APPLIANCES_REVIEWS}...")
    try:
        # We scan just the ID column to get a master list of products
        df_universe = pl.scan_parquet(S3_RAW_APPLIANCES_REVIEWS) \
                        .select("parent_asin") \
                        .unique() \
                        .collect()
        
        print(f"✅ Found {df_universe.height} unique Appliances products.")
    except Exception as e:
        print(f"❌ Error loading raw reviews: {e}")
        return

    # --- STEP 2: CALCULATE KPIs ---
    print(f"2. Loading AI Risk Data & Calculating Metrics...")
    try:
        df_tagged = pl.read_parquet(S3_TAGGED_REVIEWS)
        
        # Ensure types for math
        df_tagged = df_tagged.with_columns([
            pl.col("rating").cast(pl.Float64, strict=False),
            pl.col("timestamp").cast(pl.Int64, strict=False)
        ])
        
        # Bayesian Smoothing Constant
        C_PRIOR = 3.0
        
        product_kpis = df_tagged.group_by("parent_asin").agg([
            # Context
            pl.len().alias("analyzed_review_count"),
            
            # KPI 1: Risk Probability (Smoothed)
            ((pl.col("is_high_risk").sum() + 1) / (pl.len() + C_PRIOR)).alias("risk_probability"),
            
            # KPI 2: Dominant Driver (Excluding 'General Risk')
            pl.col("risk_category")
              .filter(pl.col("risk_category") != "General Risk")
              .mode().first()
              .alias("dominant_risk_driver"),
              
            # KPI 3: Defect Count (Absolute # of bad reviews)
            pl.col("is_high_risk").sum().alias("defect_count"),
            
            # KPI 4: Sentiment Velocity (Trend)
            pl.corr("rating", "timestamp").fill_nan(0.0).alias("sentiment_velocity")
        ])
        print("✅ KPIs Calculated.")
        
    except Exception as e:
        print(f"❌ Error processing risk data: {e}")
        return

    # --- STEP 3: JOIN & CLEAN ---
    print("3. Joining Universe with KPIs...")
    
    # Left Join: Keep ALL products. If a product wasn't in the risk sample, it gets nulls.
    df_final = df_universe.join(product_kpis, on="parent_asin", how="left")
    
    # Fill Nulls (Default to "Safe" for products we didn't flag)
    df_final = df_final.with_columns([
        pl.col("risk_probability").fill_null(0.0),
        pl.col("dominant_risk_driver").fill_null("Low Risk / No Defects"),
        pl.col("sentiment_velocity").fill_null(0.0),
        pl.col("defect_count").fill_null(0),
        pl.col("analyzed_review_count").fill_null(0)
    ])

    # --- STEP 4: EXPORT ---
    print(f"4. Saving {df_final.height} rows to CSV: {S3_FINAL_OUTPUT}...")
    df_final.write_csv(S3_FINAL_OUTPUT)
    
    print("🔥 SUCCESS! Lean KPI file ready.")
    print("Use 'parent_asin' to join this with your Metadata table in Power BI.")
    
    # Preview
    print("\n--- DATA PREVIEW ---")
    print(df_final.head(5))

# Execute
create_lean_kpi_table()

1. Loading Product Universe from s3://curated-review-data/reviews/musical_instruments_reviews.parquet...
✅ Found 190721 unique Appliances products.
2. Loading AI Risk Data & Calculating Metrics...
✅ KPIs Calculated.
3. Joining Universe with KPIs...
4. Saving 190721 rows to CSV: s3://curated-review-data/processed/kpis only/musical_risk_kpis_only.csv...
🔥 SUCCESS! Lean KPI file ready.
Use 'parent_asin' to join this with your Metadata table in Power BI.

--- DATA PREVIEW ---
shape: (5, 6)
┌─────────────┬─────────────────┬─────────────────┬────────────────┬──────────────┬────────────────┐
│ parent_asin ┆ analyzed_review ┆ risk_probabilit ┆ dominant_risk_ ┆ defect_count ┆ sentiment_velo │
│ ---         ┆ _count          ┆ y               ┆ driver         ┆ ---          ┆ city           │
│ str         ┆ ---             ┆ ---             ┆ ---            ┆ i32          ┆ ---            │
│             ┆ u32             ┆ f64             ┆ str            ┆              ┆ f64            │
╞═══

In [31]:
target_cols = [
    "parent_asin",    
    "risk_probability",     # KPI 1 (Smoothed)
    "dominant_risk_driver", # KPI 2
    "sentiment_velocity",   # KPI 3
    "defect_count",         # KPI 4
    "analyzed_review_count" # Context
]

available_cols = [c for c in target_cols if c in df_master.columns]

# 3. Select and Sort
# We sort by 'defect_count' so you see the biggest problems first
final_view = df_master.select(available_cols).filter(
    pl.col("defect_count") > 0
).sort("risk_probability", descending=True)

# 4. Print with Full Width (So titles/reasons don't get cut off)
print(f"Showing top 20 High-Risk Products (out of {final_view.height} total risks):")
with pl.Config(tbl_cols=-1, tbl_rows=20, fmt_str_lengths=50):
    print(final_view.head(20))

ColumnNotFoundError: unable to find column "defect_count"; valid columns: ["parent_asin"]